In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load Data
orders_df = pd.read_csv('../data/raw/quickcommerce_orders.csv')

# 2. Phase 2: Data Cleaning (Type casting & Binarization)
orders_df['Order Date & Time'] = pd.to_datetime(orders_df['Order Date & Time'], format='%M:%S.%f', errors='coerce')
orders_df['Delivery Delay'] = orders_df['Delivery Delay'].map({'Yes': 1, 'No': 0})
orders_df['Refund Requested'] = orders_df['Refund Requested'].map({'Yes': 1, 'No': 0})

# 3. Phase 3: Feature Engineering (Temporal & OHE)
orders_df['Order_Minute'] = orders_df['Order Date & Time'].dt.minute
orders_df = orders_df.drop(columns=['Order Date & Time'])
orders_df = pd.get_dummies(orders_df, columns=['Platform', 'Product Category'], drop_first=True)

# Cast booleans to int for correlation math
orders_df = orders_df.astype({col: 'int' for col in orders_df.select_dtypes(include='bool').columns})



In [ ]:
# Select only numerical columns for correlation math
numeric_cols = orders_df.select_dtypes(include=['int64', 'float64', 'int32']).columns
corr_matrix = orders_df[numeric_cols].corr()

# Plotting the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix: Q-Commerce Orders")
plt.show()

In [ ]:


# 1. Load Data
sales_df = pd.read_csv('../data/raw/blinkit_sales.csv')

# 2. Phase 2: Data Cleaning & Imputation
# Using modern Pandas approach to avoid the Copy-on-Write (ChainedAssignment) warning
sales_df['offer_type'] = sales_df['offer_type'].fillna("No Offer")

# 3. Proactive Type Casting for ML
# Converting boolean 'is_organic' (True/False) to integers (1/0) for future math compatibility
if 'is_organic' in sales_df.columns:
    sales_df['is_organic'] = sales_df['is_organic'].astype(int)

print("Memory State Restored! sales_df is cleaned and ready.")
# Checking the info to verify no nulls in offer_type
sales_df[['offer_type', 'is_organic']].info()

PHASE 4

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Isolate Target (y)
y = orders_df['Delivery Delay']

# 2. Isolate Features (X) by dropping leaky, non-predictive, AND text columns
# FIXED: Added 'Customer Feedback' to the drop list
leaky_cols = ['Delivery Delay', 'Delivery Time (Minutes)', 'Service Rating', 'Refund Requested', 'Order ID', 'Customer ID', 'Customer Feedback']
X = orders_df.drop(columns=leaky_cols)

# 3. Train-Test Split with Stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Define the continuous columns that need scaling
continuous_cols = ['Order Value (INR)', 'Order_Minute']

# Fit & Transform on TRAINING data only
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])

# ONLY Transform on TESTING data (using the mean/variance learned from training)
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])


print("Scaling applied strictly to continuous variables without data leakage!")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and train the Baseline Model
# Initialize and train the Baseline Model WITH Cost-Sensitive Learning
baseline_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

# Fit and Predict again...
# Print Classification Report...
baseline_model.fit(X_train, y_train)

# Predict on the unseen Test data
y_pred = baseline_model.predict(X_test)

# Print the evaluation report
print("--- Baseline Logistic Regression Performance ---")
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the mathematical constraints (Hyperparameter Grid)
# We are forcing the trees to be shallow, learn slowly, and use fractional data.
param_grid = {
    'max_depth': [3, 4, 5], 
    'learning_rate': [0.01, 0.05, 0.1], 
    'subsample': [0.7, 0.8, 0.9], 
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# 2. Initialize Base XGBoost with your calculated ratio
xgb_base = XGBClassifier(scale_pos_weight=6.31, random_state=42)

# 3. Setup the Search (Focusing explicitly on optimizing the F1-score)
# cv=3 means it breaks training data into 3 chunks for robust cross-validation
random_search = RandomizedSearchCV(
    estimator=xgb_base, 
    param_distributions=param_grid,
    n_iter=10,           # Test 10 random combinations
    scoring='f1',        # Don't optimize for accuracy, optimize for F1!
    cv=3, 
    random_state=42, 
    n_jobs=-1            # Use all CPU cores for speed
)

print("Tuning the model... This might take 1-2 minutes...")

# 4. Fit the Search
random_search.fit(X_train, y_train)

# 5. Extract the Best Model and Evaluate
print(f"Best Parameters Found: {random_search.best_params_}")

best_xgb = random_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test)

print("\n--- Tuned XGBoost Performance ---")
print(classification_report(y_test, y_pred_tuned))

In [ ]:
import ssl
import nltk

# 1. Bypass macOS SSL certificate verification
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# 2. Download stopwords safely
print("Downloading NLTK stopwords...")
nltk.download('stopwords')

print("SSL Bypass successful! Stopwords are ready to use.")

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

# Download standard NLTK stopwords (Run this once)
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# 1. Re-Load Raw Data (since we need the original text and 'Yes/No' mapping back)
raw_orders = pd.read_csv('../data/raw/quickcommerce_orders.csv')

# 2. Filter for SLA Breaches ONLY
# Note: Raw data has 'Yes'/'No', so we filter by 'Yes'
delayed_orders_df = raw_orders[raw_orders['Delivery Delay'] == 'Yes'].copy()

# Drop rows where feedback is accidentally missing
delayed_orders_df = delayed_orders_df.dropna(subset=['Customer Feedback'])

# 3. The NLP Cleaning Function
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove punctuation and special characters using Regex (keep only words and spaces)
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize by splitting spaces, filter out stopwords, and rejoin
    words = [word for word in text.split() if word not in stop_words]
    return ' '.join(words)

# 4. Apply the function to create a new column
delayed_orders_df['Cleaned Feedback'] = delayed_orders_df['Customer Feedback'].apply(clean_text)

# 5. Output Verification
print(f"Total delayed orders to analyze: {len(delayed_orders_df)}")
print("\n--- Before vs After Cleaning ---")
for index in range(3):  # Let's check the first 3 examples
    print(f"Original: {delayed_orders_df['Customer Feedback'].iloc[index]}")
    print(f"Cleaned : {delayed_orders_df['Cleaned Feedback'].iloc[index]}\n")

In [29]:
import os
from dotenv import load_dotenv   # <--- YEH LINE MISSING THI
from groq import Groq
import nltk
from nltk.corpus import stopwords

# 1. Load the environment variables right after importing
load_dotenv()

# 2. FIXING THE NLP BUG: Preserve Negations
stop_words = set(stopwords.words('english'))
# Removing negation words from the stopword list so they are NOT deleted
negation_words = {'not', 'no', 'nor', 'against', 'never', 'none', 'cannot', 'isnt', 'didnt', 'wasnt'}
custom_stop_words = stop_words - negation_words

# Re-define cleaning function
def clean_text_fixed(text):
    text = text.lower()
    words = [word for word in str(text).split() if word not in custom_stop_words]
    return ' '.join(words)

# Apply fixed cleaning
delayed_orders_df['Cleaned Feedback'] = delayed_orders_df['Customer Feedback'].apply(clean_text_fixed)

# 3. SAMPLING FOR API ECONOMICS
# Taking a random sample of 150 feedbacks to stay well within token limits
sample_feedbacks = delayed_orders_df['Cleaned Feedback'].sample(n=150, random_state=42).tolist()
combined_text_block = "\n".join(sample_feedbacks)

# 4. GROQ API (LLAMA-3) INTEGRATION
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# The Prompt Engineering
system_prompt = """You are an elite Supply Chain Operations Analyst at Blinkit/Amazon.
Your task is to perform Root Cause Analysis (RCA) on delayed order feedback.
Analyze the provided batch of customer feedbacks and output a structured report.
Identify the TOP 3 root causes for delays. For each cause, provide:
1. Cause Name
2. Percentage impact (estimate based on occurrence)
3. A 1-sentence operational solution to fix it.
Keep it strictly professional and formatted."""

print("Calling Groq API (Llama-3)... Please wait...\n")

chat_completion = client.chat.completions.create(
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Here is the cleaned feedback data:\n\n{combined_text_block}"}
    ],
    model="openai/gpt-oss-20b", 
    temperature=0.2, 
)

# 5. Print the AI Dashboard
print("====== AI ROOT CAUSE ANALYSIS DASHBOARD ======\n")
print(chat_completion.choices[0].message.content)

Calling Groq API (Llama-3)... Please wait...

====== AI ROOT CAUSE ANALYSIS DASHBOARD ======




In [30]:
# Check if we have data before sending
print(f"Text block length (characters): {len(combined_text_block)}")

if len(combined_text_block) < 50:
    print("❌ ERROR: Feedback block too small or empty!")
else:
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Analyze this:\n\n{combined_text_block}"}
            ],
            model="llama-3.3-70b-versatile",
            temperature=0.2,
        )
        # Check if message exists
        if chat_completion.choices[0].message.content:
            print("====== AI ROOT CAUSE ANALYSIS DASHBOARD ======\n")
            print(chat_completion.choices[0].message.content)
        else:
            print("❌ Model returned empty content.")
            
    except Exception as e:
        print(f"❌ API CALL FAILED: {e}")

Text block length (characters): 3617
====== AI ROOT CAUSE ANALYSIS DASHBOARD ======

**Root Cause Analysis (RCA) Report: Delayed Order Feedback**

After analyzing the provided batch of customer feedback, the top 3 root causes for delays are identified as follows:

1. **Cause Name:** Wrong Item Delivered
**Percentage Impact:** 23.1% (estimated based on occurrence)
**Operational Solution:** Implement a double-check process at the warehouse to ensure accurate item picking and packing to minimize incorrect deliveries.

2. **Cause Name:** Late Delivery
**Percentage Impact:** 14.5% (estimated based on occurrence)
**Operational Solution:** Optimize delivery routes and schedules to reduce transit times, and consider implementing a real-time tracking system to improve delivery estimates and customer communication.

3. **Cause Name:** Items Missing from Order
**Percentage Impact:** 10.3% (estimated based on occurrence)
**Operational Solution:** Conduct regular inventory audits and implement a th